# Stock Check Process Solutions

In [ ]:
import kafka

import time

import numpy as np

import json

import pprint

from IPython.display import clear_output


## You try it - 

## Modify stock_check_process to publish to the stock_check_pub_sub topic; create a new jupyter notebook stock_check_lambda_batch to subscribe to stock_check_pub_sub in batch mode; solutions can be found in stock_check_process_solutions and stock_check_lambda_batch_solutions



In [ ]:
producer = kafka.KafkaProducer(bootstrap_servers=['kafka:29092'])


In [ ]:
topic = "stock_check_queue"

stock_check_queue = kafka.KafkaConsumer(topic, 
                                       bootstrap_servers=['kafka:29092'], 
                                       auto_offset_reset='earliest')


In [ ]:
for message in stock_check_queue:
    
    clear_output(wait=True)
    
    message_dict = json.loads(message.value)
    
    print("\n=================================")
    print("   Stock Check Process")
    print("=================================\n")
    print("---------------------------------------------")
    print("   Offset:", message.offset, "   Order ID:", message_dict["order_id"])
    print("---------------------------------------------")
    
    fulfillment_list = []
    
    back_order_list = []
    
    for item in message_dict["line_items"]:
        
        if item["product_id"] == 2:
            
            back_order_list.append(item)
            
        else:
            
            fulfillment_list.append(item)
    
    if len(fulfillment_list) > 0:
        
        
        sub_total = 0
        
        for f in fulfillment_list:
            sub_total += f["total"]
            
        sub_total = round(sub_total, 2)
        
        tax = round(sub_total * 0.08, 2)
        
        total = round(sub_total + tax, 2)
             
        fulfillment = { "order_id": message_dict["order_id"],
                        "sub_total": sub_total,
                        "tax": tax,
                        "total": total,
                        "line_items": fulfillment_list}
        print("\nIn stock items:\n")
        pprint.pprint(fulfillment, sort_dicts=False)
        
        producer.send("fulfillment_queue", json.dumps(fulfillment).encode())
        
        producer.send("stock_check_pub_sub", json.dumps(fulfillment).encode())
    
    if len(back_order_list) > 0:
        
        sub_total = 0
        
        for b in back_order_list:
            sub_total += b["total"]
            
        sub_total = round(sub_total, 2)
        
        tax = round(sub_total * 0.08, 2)
        
        total = round(sub_total + tax, 2)
        
        back_order = { "order_id": message_dict["order_id"],
                       "sub_total": sub_total,
                       "tax": tax,
                       "total": total,
                       "line_items": back_order_list}
        print("\nBack Order Items:\n")
        pprint.pprint(back_order, sort_dicts=False)
        
        producer.send("back_order_queue", json.dumps(back_order).encode())
    
    time.sleep(2.0)

## You try it - 

## Modify stock_check_process to stream to the stock_check_stream topic; create a new jupyter notebook stock_check_lambda_speed to subscribe to stock_check_stream in streaming mode; solutions can be found in stock_check_process_solutions and stock_check_lambda_speed_solutions



In [ ]:
for message in stock_check_queue:
    
    clear_output(wait=True)
    
    message_dict = json.loads(message.value)
    
    print("\n=================================")
    print("   Stock Check Process")
    print("=================================\n")
    print("---------------------------------------------")
    print("   Offset:", message.offset, "   Order ID:", message_dict["order_id"])
    print("---------------------------------------------")
    
    fulfillment_list = []
    
    back_order_list = []
    
    for item in message_dict["line_items"]:
        
        if item["product_id"] == 2:
            
            back_order_list.append(item)
            
        else:
            
            fulfillment_list.append(item)
    
    if len(fulfillment_list) > 0:
        
        
        sub_total = 0
        
        for f in fulfillment_list:
            sub_total += f["total"]
            
        sub_total = round(sub_total, 2)
        
        tax = round(sub_total * 0.08, 2)
        
        total = round(sub_total + tax, 2)
             
        fulfillment = { "order_id": message_dict["order_id"],
                        "sub_total": sub_total,
                        "tax": tax,
                        "total": total,
                        "line_items": fulfillment_list}
        print("\nIn stock items:\n")
        pprint.pprint(fulfillment, sort_dicts=False)
        
        producer.send("fulfillment_queue", json.dumps(fulfillment).encode())
        
        producer.send("stock_check_pub_sub", json.dumps(fulfillment).encode())
        
        producer.send("stock_check_stream", json.dumps(fulfillment).encode())
    
    if len(back_order_list) > 0:
        
        sub_total = 0
        
        for b in back_order_list:
            sub_total += b["total"]
            
        sub_total = round(sub_total, 2)
        
        tax = round(sub_total * 0.08, 2)
        
        total = round(sub_total + tax, 2)
        
        back_order = { "order_id": message_dict["order_id"],
                       "sub_total": sub_total,
                       "tax": tax,
                       "total": total,
                       "line_items": back_order_list}
        print("\nBack Order Items:\n")
        pprint.pprint(back_order, sort_dicts=False)
        
        producer.send("back_order_queue", json.dumps(back_order).encode())
    
    time.sleep(2.0)